# Lab 3: Ubuntu Linux Essentials (Part 2)

**Duration:** 3 hours (30 min briefing, 2 h practical, 30 min check and review). **Level:** beginner. Labs 1 and 2 completed.

| Student name | Station no. | Date |
|---|---|---|
| | | |

(Double-click this cell to type your name.)

## 1. Overview
In Lab 2 you learned to move around and manage files. Now you learn who may do what on the system, how to install software, how the Jetson talks to the network, and how to watch what it is doing. You already used SSH to start this session: in this lab you learn what it does.

**By the end of this lab you will be able to:**
- Read the permission string of a file from `ls -l`.
- Change permissions with `chmod` and make a script executable.
- Explain what `sudo` does and when it is safe to use.
- Update the package list, then search, install and remove software with `apt`.
- Combine commands with a pipe (`|`) and filter output with `grep`.
- Find the Jetson's IP address, and copy files between your laptop and the Jetson with `scp`.
- Monitor CPU, memory, disk and GPU use with `top`, `htop`, `free`, `df` and `tegrastats`.

**Prerequisites:** Labs 1 and 2 completed. Your laptop and the Jetson are on the same network.

## 2. How Commands Run in This Notebook
- A cell that starts with **`%%bash`** is a small Linux terminal session. Every line is a command, run in order. A `cd` lasts until the end of that cell.
- `$LAB_HOME` is your own folder, `~/students/<your name>`.
- Commands that need `sudo` work after you run the password cell below once.
- Programs that fill the screen and wait for keys (`top`, `htop`, `less`) need a real terminal. Open one inside JupyterLab with **File, New, Terminal**, and drag its tab beside this notebook.
- Some commands are **meant to fail** (for example reading a protected file). The cell then prints a line that says so. That is the lesson, not a mistake.

### Allow `sudo`
Run this cell once and type your password in the box (nothing is shown).

In [ ]:
import labview
labview.sudo_login()

## 3. Background

### 3.1 Users, Groups and Permissions
Linux is a multi-user system. Every file has an owner, a group and a rule for everyone else ("others"). Each of the three can be allowed to read, write or execute the file. `ls -l` shows these rules in a ten-character string:

`-rwxr-xr-- 1 jetson jetson 52 Nov  1 10:00 hello.sh`

| Part | Example | Meaning |
|---|---|---|
| File type | `-` | `-` is a file, `d` is a directory, `l` is a link |
| Owner permissions | `rwx` | The owner can read, write and execute |
| Group permissions | `r-x` | Group members can read and execute, not write |
| Others permissions | `r--` | Everyone else can only read |
| Owner and group | `jetson jetson` | The user and the group that own the file |
| Size and date | `52`, `Nov 1 10:00` | Size in bytes and the last change time |
| Name | `hello.sh` | The file name |

| Letter | On a file | On a directory |
|---|---|---|
| `r` (read) | View the contents | List the names inside |
| `w` (write) | Change the contents | Create and delete files inside |
| `x` (execute) | Run it as a program or script | Enter it with `cd` |

Permissions can also be written as numbers. Read is 4, write is 2 and execute is 1. Add them for each of the three groups.

| Mode | Letters | Typical use |
|---|---|---|
| 644 | `rw-r--r--` | Normal text files. The owner edits, everyone can read. |
| 755 | `rwxr-xr-x` | Scripts and programs. Everyone can run, only the owner edits. |
| 600 | `rw-------` | Private files. Only the owner has access. |
| 700 | `rwx------` | Private scripts or folders. |

### 3.2 What Is `sudo`?
One special user, `root`, can do anything on the system, including deleting the operating system. Your normal account cannot. `sudo` ("superuser do") runs one command with root rights.
- Think before you press Enter. A mistake made with `sudo` can break the board.
- Never run a `sudo` command you do not understand.
- Do not use `sudo` to get around "Permission denied" for files in your own home folder. Fix the permissions instead.

### 3.3 Software Packages and `apt`
Linux software comes as packages stored on online servers called repositories. `apt` finds a package, downloads it, installs it and also installs anything it depends on. NVIDIA runs its own repository for the JetPack packages.

| Command | What it does |
|---|---|
| `sudo apt update` | Refreshes the list of available packages. It does not install anything. |
| `apt search word` | Searches package names and descriptions |
| `apt show name` | Shows details about one package |
| `sudo apt install name` | Installs a package and its dependencies |
| `sudo apt remove name` | Removes a package |
| `sudo apt upgrade` | Upgrades every installed package. Not used in this course unless your instructor says so. |

### 3.4 Networks and SSH
Every device on a network has an IP address, such as `192.168.1.50`. SSH (Secure Shell) opens a terminal on another computer over the network, encrypted. SCP copies files over the same secure connection.

```
Your laptop                               The Jetson
ssh orin_nano@192.168.1.50  ------------->  SSH server (listens on port 22)
(terminal on the laptop)                  commands you type run HERE
```

### 3.5 Pipes and `grep`
A pipe (`|`) sends the output of one command into the next one. For example `dpkg -l` lists every installed package, which is hundreds of lines. Piping it into other tools makes it useful.

| Command | What it does |
|---|---|
| `dpkg -l \| wc -l` | Counts the lines (`wc -l`), so it counts the installed packages |
| `dpkg -l \| grep python3` | `grep` keeps only the lines that contain `python3` |
| `dpkg -l \| grep -i nvidia` | Same, ignoring upper and lower case (`-i`) |
| `dpkg -l \| head -n 5` | Shows only the first five lines |

## 4. Procedure

> **Tip.** Keep all your work for this lab in `$LAB_HOME/lab3`. Press the stop button (square) in the notebook toolbar to stop a cell that seems stuck.

## Part A: Permissions and sudo (30 min)

### Step 1: Create the Lab 3 Workspace

In [ ]:
%%bash
cd "$LAB_HOME"
mkdir -p lab3
cd lab3
mkdir -p scripts files
ls -l

### Step 2: Read Permission Strings
Lab 2 made `board_info.txt`. If it is missing, the first command shows an error.

In [ ]:
%%bash
ls -l "$LAB_HOME/lab2/notes/board_info.txt"
ls -l /etc/hostname
ls -l /etc/shadow
ls -l /bin/ls

The next command should fail with `Permission denied`. That is the system protecting the password file.

In [ ]:
%%bash
cat /etc/shadow || echo "(failed, as expected: your account may not read this file)"

| File | Permission string | Owner | Can you read it? |
|---|---|---|---|
| `board_info.txt` | | | |
| `/etc/hostname` | | | |
| `/etc/shadow` | | | |
| `/bin/ls` | | | |

### Step 3: Make a Script Executable
A script is a text file of commands. This cell creates `scripts/hello.sh` with four lines. The first line, `#!/bin/bash`, tells Linux which program should run the file. The `$(...)` parts run a command and insert its output into the text.

In [ ]:
%%bash
cd "$LAB_HOME/lab3"
printf '#!/bin/bash\necho "Hello from the Jetson"\necho "Today is $(date)"\necho "I am $(whoami)"\n' > scripts/hello.sh
chmod -x scripts/hello.sh
cat scripts/hello.sh

Now try to run it. The `./` means "in this folder". It should fail, because a new file has no execute permission.

In [ ]:
%%bash
cd "$LAB_HOME/lab3"
./scripts/hello.sh || echo "(failed, as expected: no x in the permission string)"
ls -l scripts/hello.sh

`chmod +x` adds the execute permission. The `x` appears in the permission string and the script runs.

In [ ]:
%%bash
cd "$LAB_HOME/lab3"
chmod +x scripts/hello.sh
ls -l scripts/hello.sh
./scripts/hello.sh

### Step 4: Use Numeric Permissions

In [ ]:
%%bash
cd "$LAB_HOME/lab3"
echo "my secret" > files/secret.txt
ls -l files/secret.txt
chmod 600 files/secret.txt
ls -l files/secret.txt
chmod 644 files/secret.txt
ls -l files/secret.txt

Now write the `chmod` command for each goal in the cell below it, and check each with `ls -l`.

| Goal | Your command |
|---|---|
| `files/secret.txt`: owner can read and write, nobody else has any access | |
| `scripts/hello.sh`: everyone can run it, only the owner can edit it | |
| `scripts/hello.sh`: take away the write permission for the group (use letters, not numbers) | |
| `files/secret.txt`: everyone can read it but nobody can write to it | |

In [ ]:
%%bash
cd "$LAB_HOME/lab3"
# type your four chmod commands here, each followed by an ls -l line

### Step 5: Try `sudo`
`sudo whoami` prints `root`, which shows that the command ran as root. `ls /root` is refused, but `sudo ls /root` works.

> **Warning.** Do not put `sudo` in front of `rm`, `chmod` or `chown` on folders such as `/`, `/etc`, `/usr` or `/home`. One wrong path can make the board unbootable.

In [ ]:
%%bash
whoami
sudo whoami
ls /root || echo "(refused, as expected)"
sudo ls /root

## Part B: Install Software with apt (30 min)

### Step 1: Update the Package List
Read the last lines. A message such as "packages can be upgraded" is normal.

> **Warning.** Do not run `sudo apt upgrade` or `sudo apt full-upgrade`. They can replace JetPack components and break the setup used in later labs.

In [ ]:
!sudo apt update

### Step 2: Search and Inspect a Package

In [ ]:
%%bash
apt search htop 2>/dev/null | head -n 10
echo "----"
apt show htop 2>/dev/null | head -n 8

### Step 3: Install and Run
The setup in Lab 0 already installed `htop`, so `apt` may say that it is already the newest version. That is fine. `-y` answers "yes" to the questions.

In [ ]:
!sudo apt install htop -y

`htop` fills the screen with a live view, so it needs a real terminal. Print its version here, then type `htop` in the **Terminal tab**. Press `q` to quit.

In [ ]:
!htop --version

### Step 4: Remove and Reinstall

In [ ]:
%%bash
sudo apt remove htop -y
htop --version || echo "(command not found, as expected after the removal)"

Reinstall it, because you need it in Part D:

In [ ]:
!sudo apt install htop -y

### Step 5: Explore Installed Packages with Pipes

In [ ]:
%%bash
dpkg -l | wc -l
echo "---- python3"
dpkg -l | grep -i python3 | head -n 5
echo "---- nvidia"
dpkg -l | grep -i nvidia | head -n 10
echo "---- jetpack"
dpkg -l | grep nvidia-jetpack

| Command | What it shows | Your result |
|---|---|---|
| `dpkg -l \| wc -l` | How many lines (about how many packages) | |
| `dpkg -l \| grep -i nvidia \| head -n 10` | The first ten NVIDIA packages | |
| `dpkg -l \| grep nvidia-jetpack` | Whether the JetPack bundle package is installed | |

### Step 6: Look at NVIDIA's Repository
The lines that start with `deb` give the web addresses where JetPack packages are downloaded from.

In [ ]:
%%bash
ls /etc/apt/sources.list.d/
cat /etc/apt/sources.list.d/nvidia*

## Part C: Network and SSH (35 min)

### Step 1: Find Your Address and Test the Network
`hostname -I` may print more than one address. Use the first one that looks like `192.168.x.x` or `10.x.x.x`. `-c 4` makes `ping` send four packets and stop. The first `ping` tests the connection. The second also tests name lookup (DNS).

In [ ]:
%%bash
hostname
hostname -I
ping -c 4 8.8.8.8
ping -c 4 google.com

| Hostname | |
|---|---|
| IP address | |
| Username | |

### Step 2: Check the SSH Server
Look for a green `active (running)`.

In [ ]:
!sudo systemctl status ssh --no-pager

| What you see | What to run |
|---|---|
| `inactive` or `dead` | `sudo systemctl enable --now ssh` |
| `Unit ssh.service could not be found` | `sudo apt install openssh-server -y` |

### Step 3: Connect from Your Laptop
You did this at the start of the session, to run `lab start`. This cell prints the exact line. On your laptop (Terminal on macOS and Linux, PowerShell on Windows) it looks like this:

In [ ]:
%%bash
echo "    ssh $(whoami)@$(hostname -I | awk '{print $1}')"

1. The first time, SSH asks whether to trust the computer. Type `yes`.
2. Type the Jetson password. Nothing appears while you type it.
3. The prompt shows the Jetson's name. Every command you type runs on the Jetson, not on your laptop.
4. Type `exit` to disconnect. (Do **not** do this now: it ends your session. Just remember it.)

> **Tip.** If your network supports it, try the name instead of the number, for example `ssh orin_nano@jetson-01.local`. This keeps working even when the IP address changes.

### Step 4: Copy Files with `scp`
Run these on your laptop, not on the Jetson. The cell prints the lines with your own user, address and folder:

In [ ]:
%%bash
IP=$(hostname -I | awk '{print $1}')
echo '    echo "Hello from my laptop" > laptop_note.txt'
echo "    scp laptop_note.txt $(whoami)@$IP:$LAB_HOME/lab3/files/"
echo "    scp $(whoami)@$IP:$LAB_HOME/lab2/notes/board_info.txt ."

The pattern is `scp source destination`. A remote location is written `user@address:path`. The dot (`.`) in the last command means "the current folder on the laptop". Then run the next cell to check that `laptop_note.txt` arrived.

In [ ]:
%%bash
ls -l "$LAB_HOME/lab3/files"

> **Note.** Passwords over SSH are fine inside a classroom network. For real deployments, use SSH keys, change default passwords and never share your login.

## Part D: Monitor the System (25 min)

### Step 1: Quick Health Checks

In [ ]:
%%bash
uptime
echo "----"
free -h
echo "----"
df -h /
echo "----"
du -sh "$LAB_HOME"

| Command | What it shows | Your reading |
|---|---|---|
| `uptime` | Time since boot, and three load averages (1, 5 and 15 minutes) | |
| `free -h` | Total, used and available memory | |
| `df -h /` | Disk size, space used and the percentage used | |
| `du -sh $LAB_HOME` | Total size of your lab folder | |

### Step 2: Watch the System Live
`top` and `htop` fill the screen and wait for keys, so use the **Terminal tab** (File, New, Terminal). Type `top` and press `q` to quit. Then type `htop`, which shows the same data with colours. The bars at the top are the six CPU cores and the memory. Press `q` or `F10` to quit.

A notebook cell can still show one snapshot of `top` (`-b` means batch mode, `-n 1` means one round):

In [ ]:
!top -b -n 1 | head -n 15

### Step 3: Find and Stop a Process
This cell starts a command that does nothing for five minutes. The `&` at the end lets it run in the background. `$!` is the process ID (PID) of the command you just started. `pgrep sleep` finds processes by name (other students on this Jetson may have their own `sleep` running, so we stop only ours with `kill $!`).

In [ ]:
%%bash
sleep 300 &
echo "The PID of my sleep is $!"
pgrep sleep
kill $!
sleep 1
ps -p $! > /dev/null || echo "My sleep process is gone."

Write the PID you saw: ______________

### Step 4: Watch the Jetson with `tegrastats`
One line prints every second. `timeout 10` stops it by itself after 10 seconds. (In a terminal you stop it with Ctrl + C.)

In [ ]:
!sudo timeout 10 tegrastats

| Field | Meaning |
|---|---|
| `RAM used/total` | Memory in use out of the memory the programs can use, in MB. The CPU and GPU share it. |
| `SWAP` | Disk space used as extra memory. It should stay near zero. |
| `CPU [x%@MHz, ...]` | Load on each core and its current speed |
| `GR3D_FREQ x%` | How busy the GPU is. This is the number to watch in AI labs. |
| Temperatures (`...@45C`) | Temperature of each part, in degrees Celsius |
| Power values (`VDD...`) | Power drawn by parts of the board, in mW |

Field names can differ slightly between JetPack versions. Open jtop from Lab 1 in the Terminal tab and find the same RAM, CPU and GPU numbers. They come from the same data.

### Step 5: See the Effect of Load
First read the **idle** values (the board is doing nothing):

In [ ]:
!top -b -n 1 -1 | head -n 14

Now `yes > /dev/null &` starts a command that keeps one CPU core busy. The cell waits 30 seconds, reads the values again, and stops the command.

In [ ]:
%%bash
yes > /dev/null &
sleep 30
top -b -n 1 -1 | head -n 14
kill $!
echo "yes is stopped."

| Condition | Busiest core load (%) | Temperature (C) |
|---|---|---|
| Idle | | |
| With `yes` running | | |

### Step 6: Restart and Shut Down Safely
Use only one of these. They end your session. Save the notebook first (**Ctrl + S**). `reboot` restarts the board. `shutdown now` switches it off. Wait for the fan and the lights to stop before removing power.

In [ ]:
# !sudo reboot
# !sudo shutdown now
# remove the # at the start of ONE line to run it

> **Challenge (optional).** If you have a USB camera, run `ls /dev/video*` and `lsusb` in a cell. Plug in the camera and run both again. What changed? You use this camera in Lab 6.

## 5. Check Task
Show the following to your instructor. The instructor initials each line when it is done correctly.

1. `./scripts/hello.sh` runs from `$LAB_HOME/lab3` and prints three lines. `ls -l scripts/hello.sh` shows the `x` permission.
2. `ls -l files/secret.txt` shows `-rw-------`.
3. `htop` opens (installed with `apt`) in the Terminal tab and shows six CPU bars.
4. From your laptop, you connect to the Jetson with `ssh` without help and run `hostname`.
5. `files/laptop_note.txt` is on the Jetson, copied with `scp`.
6. You start `sleep 300`, find its PID and stop it with `kill`.
7. The tables in Parts A, B, C and D are complete.

## 6. Assessment Questions
Answer in your own words in this cell. One or two sentences are enough unless the question says otherwise.

1. In the permission string `-rwxr-xr--`, what can the owner, the group and others each do?
2. Write `chmod 640 notes.txt` in letters. Who can do what?
3. Why does `./hello.sh` fail with `Permission denied` right after you create it? How do you fix it?
4. What does `sudo` do? Why should you avoid it for files in your own home folder?
5. What is the difference between `sudo apt update` and `sudo apt upgrade`? Why do we avoid `upgrade` in this course?
6. What does a pipe (`|`) do? Explain what `dpkg -l | grep nvidia | wc -l` tells you.
7. You can `ping 8.8.8.8` but not `ping google.com`. What is probably wrong?
8. Which command copies `a.txt` from your laptop to the Jetson? (a) `ssh a.txt orin_nano@IP` (b) `scp a.txt orin_nano@IP:~/` (c) `scp orin_nano@IP:~/ a.txt` (d) `cp a.txt orin_nano@IP`

## 7. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Permission denied` when you run a script | The file has no execute permission | Run `chmod +x filename`. |
| `Could not get lock /var/lib/dpkg/lock-frontend` | Another install or update is running | Wait a minute and try again. Do not delete lock files. |
| `Unable to locate package` or `Temporary failure resolving` | No internet, or the package list is old | Check `ping 8.8.8.8`. Run `sudo apt update`. |
| `ping 8.8.8.8` works but `ping google.com` fails | A name lookup (DNS) problem | Ask your instructor to check the network DNS setting. |
| `ssh: connect to host ... port 22: Connection refused` | The SSH server is not running | Run `sudo systemctl enable --now ssh`. |
| `ssh` hangs, then `Connection timed out` | Wrong IP, different networks, or the network blocks device-to-device traffic | Check `hostname -I` again. Ask your instructor. |
| `Permission denied, please try again` in `ssh` | Wrong username or password | Run `whoami` on the Jetson to check the username. |
| `WARNING: REMOTE HOST IDENTIFICATION HAS CHANGED!` | The board was reflashed, or another board now has that IP address | Ask your instructor. The fix is `ssh-keygen -R <IP>` on the laptop. |
| `sudo` asks for a password and the cell hangs | You did not run the `sudo_login` cell | Run it, then run the cell again. |
| A cell never finishes | The command waits for keys (`top`, `htop`, `less`) | Press the stop button (square), and use the Terminal tab. |

## 8. Further Reading
- NVIDIA Jetson Linux Developer Guide: Tegrastats Utility: https://docs.nvidia.com/jetson/archives/r36.4/DeveloperGuide/AT/JetsonLinuxDevelopmentTools/TegrastatsUtility.html
- Microsoft Learn: OpenSSH for Windows overview: https://learn.microsoft.com/windows-server/administration/openssh/openssh_overview
- Ubuntu Desktop documentation: The Linux Command Line for Beginners: https://ubuntu.com/desktop/docs/en/24.04/tutorial/the-linux-command-line-for-beginners/